In [1]:
from pathlib import Path
from dashscope import Generation
from openai import OpenAI
import os

file_path = "design/慢病随访调查问卷.pdf"  # 文件路径
QWEN_API_KEY = ""  # 替换成自己的通义的API_KEY
KIMI_API_KEY = ""
QWEN_BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"  # 填写DashScope服务endpoint
KIMI_BASE_URL = "https://api.moonshot.cn/v1"

# 初始化API客户端
qwen_client = OpenAI(api_key=QWEN_API_KEY,base_url=QWEN_BASE_URL)
kimi_client = OpenAI(api_key=KIMI_API_KEY,base_url=KIMI_BASE_URL)

Q_PROMPT = PROMPT = """
请按照以下要求，作为后续医护人员的智能助手，使用中文提出问题，并生成问卷摘要：
1. You will play the role of a follow-up healthcare worker, asking users questions based on the questionnaire content, and not answering questions yourself!.
2. Ask one question at a time and wait for the user to answer before moving on to the next question,.
3. Only ask questions about the first seven modules, without involving modules eight or nine.
After the Q&A, generate a questionnaire summary based on the existing questionnaire content and return it in JSON structure.
5. Strictly follow the order of the questionnaire to ask questions and ensure that no questions are missed.
6. Ensure accurate questioning and provide clear options for users to choose from.
7. If the user provides an answer that does not match the question, please continue to ask until you receive the correct answer.
8. 对民族信息只记录，不需要解析。
9. Each time you ask a question, please provide examples of options to choose from.
10. If the questionnaire is interrupted or ended, please return the summary of the questionnaire collected so far.
11. Do not make assumptions, only return the content that the user has answered.
12. The context content is too long, pay attention to controlling the output. The output can only be a problem, do not bring anything else.
13. Carefully read the above requirements and complete the task according to them.
"""


S_PROMPT = """
你是一名问卷设计人员，非常擅长根据读取到的内容生成完整的问卷。接下来你会读取到文件的内容，根据内容，你会设计出一份慢病随访访谈式问卷
要求：1.问卷顺序：以清晰的序列和模块排序每个问题。
     2.问卷格式：问卷整体简洁清晰明了，不使用过多符号。
"""

In [2]:
def perform_chat(client, model_name, messages, temperature, max_tokens):
    completion = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return completion.choices[0].message.content

def kimi_chat(system_prompt, user_content, max_tokens=5000):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]
    return perform_chat(kimi_client, "moonshot-v1-8k", messages, temperature=0.3, max_tokens=max_tokens)

def qwen_chat(system_prompt, user_content, max_tokens=2000):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]
    return perform_chat(qwen_client, "qwen-turbo", messages, temperature=0.85, max_tokens=max_tokens)


### 生成问卷

In [3]:
# try:
#     # 上传文件并创建文件对象
#     with Path(file_path).open('rb') as file:
#         file_object = kimi_client.files.create(file=file, purpose="file-extract")
#         # 获取文件内容
#         file_content = kimi_client.files.content(file_id=file_object.id).text
#         # 删除文件对象，清理资源
#         kimi_client.files.delete(file_id=file_object.id)
# except Exception as e:
#     print(f"Error during file upload and content extraction: {e}")

In [4]:
# user_content = f"这是问卷内容\"\"\"{file_content}\"\"\""
# system_prompt = S_PROMPT
# questionnaire = kimi_chat(system_prompt, user_content)

In [5]:
file1_path = 'Q.txt'
with open(file1_path, 'r', encoding='utf-8') as file:
    file_content = file.read()

In [ ]:
output_file = 'final.txt'
dialogue_history = "问卷调查开始" 
initial_prompt = f"按照以下要求来实现本次对话。\n \"\"\"{Q_PROMPT}\"\"\"。"

# questionnaire_text = f"这是问卷内容：\n \"\"\"{questionnaire}\"\"\""
questionnaire_text = f"这是问卷内容：\n \"\"\"{file_content}\"\"\""
prompt = f"{initial_prompt} {questionnaire_text}"
current_question = qwen_chat(initial_prompt, questionnaire_text)

while current_question.strip()!="":
    question_w = current_question.split(" Answer: ")[0]
    print("Question: ", question_w)
    user_response = input("Your answer: ")
    dialogue_history += f" {question_w} Answer: {user_response} \n"

    # 再次调用模型生成下一个问题
    current_question = qwen_chat(prompt, dialogue_history)
